# Multimodal Attention MIL for CAD Detection — reproducible pipeline

Run order: (1) imports + CONFIG, (2) synthetic-mode switch, (3) full pipeline.
Set `USE_SYNTHETIC = False` in cell 2 to use the real (restricted) dataset.
First run `python make_synthetic_example.py` to create `toy_data/`.


In [ ]:
# ============================================================
# SETUP CELL -- run this FIRST (once).
# Installs nibabel and generates the synthetic toy dataset
# under ./toy_data/ so the pipeline can run without the
# restricted patient data. Safe to re-run.
# ============================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel'], check=False)

# ============================================================
# make_synthetic_example.py
# Generates a SYNTHETIC toy dataset that mimics the MultiD4CAD
# directory layout, so the full MIL pipeline can be run and
# reproduced WITHOUT any access to the restricted patient data.
#
# It writes, under ./toy_data/ :
#   clinicalDataWithLabels.CSV                         (sep=";")
#   Epicardial Adipose Tissue Segmentations (118 patients)/<PID>/
#       epicardialDatasetNIFTI.nii
#       epicardialDatasetNIFTI_mask.nii
#   Pericoronaric Adipose Tissue Segmentations (118 patients)/<PID>/
#       coronaricDatasetNIFTI.nii
#       coronaricDatasetNIFTI_mask.nii
#
# The images contain random intensities and a small blob mask, which
# is enough for the 2.5D instance builder. No real anatomy, no PHI.
# ============================================================
import os
import numpy as np
import pandas as pd
import nibabel as nib

# ---- configuration of the toy set -------------------------------
OUT_ROOT   = "toy_data"
N_PATIENTS = 16          # small, both classes
VOL_SHAPE  = (64, 64, 24)  # (H, W, depth) -- depth is the slice axis
SEED       = 0
rng = np.random.default_rng(SEED)

EAT_DIR = "Epicardial Adipose Tissue Segmentations (118 patients)"
PAT_DIR = "Pericoronaric Adipose Tissue Segmentations (118 patients)"

os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(os.path.join(OUT_ROOT, EAT_DIR), exist_ok=True)
os.makedirs(os.path.join(OUT_ROOT, PAT_DIR), exist_ok=True)


def make_volume_and_mask(depth_blob_frac=0.6, blob_radius=8):
    """Random CT-like volume + a blob mask on a subset of slices."""
    H, W, D = VOL_SHAPE
    vol = rng.normal(0.0, 1.0, size=VOL_SHAPE).astype(np.float32)
    mask = np.zeros(VOL_SHAPE, dtype=np.float32)
    # put a circular blob on the central slices so mask area >= 10 px
    n_slices = max(3, int(D * depth_blob_frac))
    start = (D - n_slices) // 2
    yy, xx = np.ogrid[:H, :W]
    for k in range(start, start + n_slices):
        cy = rng.integers(blob_radius, H - blob_radius)
        cx = rng.integers(blob_radius, W - blob_radius)
        r = blob_radius + rng.integers(-2, 3)
        disk = (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2
        mask[disk, k] = 1.0
        # make the tissue inside the blob slightly brighter
        vol[disk, k] += 2.0
    return vol, mask


def save_nii(arr, path):
    nib.save(nib.Nifti1Image(arr.astype(np.float32), affine=np.eye(4)), path)


rows = []
for i in range(N_PATIENTS):
    pid = f"P{i+1:03d}"
    label = int(i % 2 == 0)  # alternate classes -> balanced

    # EAT (larger depot) and PAT (smaller depot, contained-like)
    eat_vol, eat_msk = make_volume_and_mask(depth_blob_frac=0.7, blob_radius=10)
    pat_vol, pat_msk = make_volume_and_mask(depth_blob_frac=0.5, blob_radius=6)

    eat_p = os.path.join(OUT_ROOT, EAT_DIR, pid)
    pat_p = os.path.join(OUT_ROOT, PAT_DIR, pid)
    os.makedirs(eat_p, exist_ok=True)
    os.makedirs(pat_p, exist_ok=True)

    save_nii(eat_vol, os.path.join(eat_p, "epicardialDatasetNIFTI.nii"))
    save_nii(eat_msk, os.path.join(eat_p, "epicardialDatasetNIFTI_mask.nii"))
    save_nii(pat_vol, os.path.join(pat_p, "coronaricDatasetNIFTI.nii"))
    save_nii(pat_msk, os.path.join(pat_p, "coronaricDatasetNIFTI_mask.nii"))

    # clinical row: make Age weakly informative so the pipeline is non-trivial
    age = int(rng.normal(58 if label == 1 else 64, 8))
    rows.append({
        "PatientID": pid,
        "Age": age,
        "BMI": round(float(rng.normal(27, 4)), 1),
        "Sex": rng.choice(["M", "F"]),
        "Smoking": rng.choice(["yes", "no"]),
        "Diabetes": rng.choice(["yes", "no"]),
        "Arterial hypertension": rng.choice(["yes", "no"]),
        "Hypercholesterolemia": rng.choice(["yes", "no"]),
        "Family history": rng.choice(["yes", "no"]),
        "Label (CAD=1; no CAD=0)": label,
    })

df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_ROOT, "clinicalDataWithLabels.CSV")
df.to_csv(csv_path, sep=";", index=False)

print(f"Synthetic toy dataset written under: {OUT_ROOT}/")
print(f"  patients: {N_PATIENTS} (CAD+ {int(df['Label (CAD=1; no CAD=0)'].sum())}, "
      f"CAD- {int((1-df['Label (CAD=1; no CAD=0)']).sum())})")
print(f"  clinical CSV: {csv_path}")
print(f"  EAT dir: {os.path.join(OUT_ROOT, EAT_DIR)}/<PID>/")
print(f"  PAT dir: {os.path.join(OUT_ROOT, PAT_DIR)}/<PID>/")
print("\nThis is SYNTHETIC data with no real anatomy and no patient information.")

# quick sanity check
import os
assert os.path.exists('toy_data'), 'toy_data not created'
print('toy_data ready:', sorted(os.listdir('toy_data'))[:3], '...')


In [ ]:
# ============================================================
# ✅ NOTEBOOK COMPLET (Kaggle SAFE) MultiD4CAD
# Dual MIL 2.5D (EAT+PAT) + Clinical
# + ResNet18 pretrained local (NO internet)
# + Best epoch by AUC (publication-friendly)
# + Scheduler + Early stopping
# + OOF GLOBAL: AUC-ROC + AUC-PR + Bootstrap CI95
# + Baseline Clinical-only OOF + ΔAUC (bootstrap)
# ============================================================

import os, sys, pickle, random, warnings, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, average_precision_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

import nibabel as nib

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
CONFIG = {
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),

    # Data paths (generic; see README to place the real dataset)
    "data_root": "data/MultiD4CAD/",
    "clinical_csv": "clinicalDataWithLabels.CSV",
    "patient_id_col": "PatientID",
    "label_col": "Label (CAD=1; no CAD=0)",
    "eat_base_dir": "Epicardial Adipose Tissue Segmentations (118 patients)",
    "pat_base_dir": "Pericoronaric Adipose Tissue Segmentations (118 patients)",

    # NIfTI
    "eat_img": "epicardialDatasetNIFTI.nii",
    "eat_msk": "epicardialDatasetNIFTI_mask.nii",
    "pat_img": "coronaricDatasetNIFTI.nii",
    "pat_msk": "coronaricDatasetNIFTI_mask.nii",

    # Cache
    "cache_file": "cache_mil_2_5d_dual_safe.pkl",

    # MIL 2.5D
    "target_hw": (224, 224),
    "num_instances": 24,
    "slice_thickness": 1,
    "min_mask_area": 10,

    # Training
    "num_folds": 5,
    "epochs": 30,
    "batch_size": 4,
    "lr": 2e-4,
    "weight_decay": 5e-4,
    "grad_clip": 1.0,
    "dropout": 0.5,
    "use_amp": True,

    # Ensemble
    "seeds": [42, 43, 44],

    # Threshold objective weights (utilisé seulement pour reporting)
    "w_auc": 0.35,
    "w_f1": 0.25,
    "w_rec": 0.20,
    "w_prec": 0.10,
    "w_spec": 0.10,

    # Local pretrained ResNet18 weights
    "resnet18_local_pth": "weights/resnet18-f37072fd.pth",

    "output_dir": "publication_outputs/",
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)

print("="*70)
print("✅ CONFIGURATION")
print("="*70)
print("Device:", CONFIG["device"])
print("dropout:", CONFIG["dropout"])
print("weight_decay:", CONFIG["weight_decay"])
print("epochs:", CONFIG["epochs"])
print("cache:", CONFIG["cache_file"])
print("resnet18_local_pth:", CONFIG["resnet18_local_pth"])
print("="*70)


In [ ]:
# ============================================================
# synthetic_config_patch.py
# Paste this as a NEW CELL immediately AFTER the CONFIG cell
# (the one that defines the CONFIG dictionary) in the notebook.
#
# When USE_SYNTHETIC = True, the pipeline runs on the synthetic
# toy dataset produced by make_synthetic_example.py, trains the
# CNN from scratch (no pretrained weights needed), and uses a
# small/fast setting. Set USE_SYNTHETIC = False to run on the
# real (restricted) dataset exactly as before.
# ============================================================
import os

USE_SYNTHETIC = True   # <-- set to False for the real dataset

if USE_SYNTHETIC:
    CONFIG["data_root"]          = "toy_data/"                 # from make_synthetic_example.py
    CONFIG["resnet18_local_pth"] = None                        # train from scratch
    CONFIG["cache_file"]         = "cache_toy.pkl"
    CONFIG["output_dir"]         = "toy_outputs/"
    CONFIG["epochs"]             = 3                            # fast demo
    CONFIG["seeds"]              = [42]                         # single seed for speed
    CONFIG["num_folds"]          = 3                            # small cohort
    os.makedirs(CONFIG["output_dir"], exist_ok=True)
    print("=" * 60)
    print("SYNTHETIC MODE ENABLED")
    print("  data_root :", CONFIG["data_root"])
    print("  pretrained:", CONFIG["resnet18_local_pth"], "(train from scratch)")
    print("  epochs    :", CONFIG["epochs"], "| seeds:", CONFIG["seeds"],
          "| folds:", CONFIG["num_folds"])
    print("  This is synthetic data: results are meaningless, the goal")
    print("  is only to demonstrate that the pipeline runs end to end.")
    print("=" * 60)


# ------------------------------------------------------------
# Fix for a missing helper used by the ablation table.
# The notebook calls best_threshold_by_f1(...) but never defines it.
# Define it here so the full notebook (including the ablation
# study) runs end to end without a NameError.
# ------------------------------------------------------------
import numpy as np
from sklearn.metrics import f1_score

def best_threshold_by_f1(y_true, y_prob):
    """Return (threshold, f1) maximizing F1 over a grid."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    best_thr, best_f1 = 0.5, -1.0
    for thr in np.linspace(0.05, 0.95, 19):
        f1 = f1_score(y_true, (y_prob >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = float(f1), float(thr)
    return best_thr, best_f1


In [ ]:
# ============================================================
# Utils
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def safe_znorm(x: np.ndarray, eps=1e-6):
    x = x.astype(np.float32)
    m = np.mean(x)
    s = np.std(x)
    if (not np.isfinite(s)) or s < eps:
        return np.zeros_like(x, dtype=np.float32)
    z = (x - m) / (s + eps)
    z = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
    return z.astype(np.float32)

def resize_hw(img: np.ndarray, out_hw=(224,224)):
    t = torch.tensor(img, dtype=torch.float32)[None, None]
    t = torch.nn.functional.interpolate(t, size=out_hw, mode="bilinear", align_corners=False)
    return t[0,0].cpu().numpy()

def binarize_label(y):
    return int(y) if y in [0,1] else int(float(y))

def compute_metrics(y_true, y_prob, thr=0.5):
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob).astype(float)
    y_pred = (y_prob >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    spec = tn / (tn + fp + 1e-9)

    auc_val = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    prec= precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)

    return {"auc": auc_val, "acc": acc, "f1": f1, "prec": prec, "rec": rec, "spec": spec}

def balanced_objective(m: dict):
    return (
        CONFIG["w_auc"]  * m["auc"]  +
        CONFIG["w_f1"]   * m["f1"]   +
        CONFIG["w_rec"]  * m["rec"]  +
        CONFIG["w_prec"] * m["prec"] +
        CONFIG["w_spec"] * m["spec"]
    )

def search_best_threshold(y_true, y_prob):
    best = {"thr": 0.5, "score": -1e9, "metrics": None}
    for thr in np.linspace(0.05, 0.95, 37):
        m = compute_metrics(y_true, y_prob, thr=thr)
        s = balanced_objective(m)
        if s > best["score"]:
            best = {"thr": float(thr), "score": float(s), "metrics": m}
    return best

def load_nifti(path: Path):
    img = nib.load(str(path))
    data = img.get_fdata().astype(np.float32)
    return data, img

def get_slice_mask_areas(mask3d):
    shp = mask3d.shape
    depth_axis = int(np.argmin(shp))
    m = np.moveaxis(mask3d, depth_axis, 0)
    areas = m.reshape(m.shape[0], -1).sum(axis=1)
    return m, areas, depth_axis

def _stable_int_seed(s: str) -> int:
    h = hashlib.md5(s.encode("utf-8")).hexdigest()
    return int(h[:8], 16)

def build_2p5d_instances(vol3d, msk3d, num_instances, out_hw, patient_id="NA"):
    m, areas, depth_axis = get_slice_mask_areas(msk3d)
    v = np.moveaxis(vol3d, depth_axis, 0)
    D = v.shape[0]

    cand = np.where(areas >= CONFIG["min_mask_area"])[0].tolist()
    if len(cand) == 0:
        cand = list(range(max(1, D//4), min(D-1, 3*D//4)))

    cand_sorted = sorted(cand, key=lambda i: float(areas[i]), reverse=True)
    top_pool = cand_sorted[: max(num_instances * 3, num_instances)]

    rng = np.random.default_rng(_stable_int_seed(str(patient_id)))

    if len(top_pool) >= num_instances:
        chosen = rng.choice(top_pool, size=num_instances, replace=False).tolist()
    else:
        chosen = rng.choice(top_pool, size=num_instances, replace=True).tolist()

    chosen = sorted(chosen)

    inst = []
    for i in chosen:
        i0 = max(0, i - CONFIG["slice_thickness"])
        i1 = i
        i2 = min(D-1, i + CONFIG["slice_thickness"])

        s0, s1, s2 = v[i0], v[i1], v[i2]
        ms = m[i]

        s0 = safe_znorm(s0 * ms)
        s1 = safe_znorm(s1 * ms)
        s2 = safe_znorm(s2 * ms)

        s0 = resize_hw(s0, out_hw)
        s1 = resize_hw(s1, out_hw)
        s2 = resize_hw(s2, out_hw)

        x = np.stack([s0, s1, s2], axis=0).astype(np.float32)
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        inst.append(x)

    return np.stack(inst, axis=0)

# ============================================================
# Clinical Processor
# ============================================================
class ClinicalProcessor:
    def __init__(self):
        self.numeric_cols = []
        self.cat_cols = []
        self.scaler = None
        self.ohe = None
        self.feature_names_ = None

    def fit(self, df_train: pd.DataFrame, id_col: str, label_col: str):
        df = df_train.copy()
        df.columns = [c.strip().replace('"','') for c in df.columns]

        if id_col not in df.columns:
            cand = [c for c in df.columns if c.lower().strip() == id_col.lower().strip()]
            if len(cand) == 1:
                df = df.rename(columns={cand[0]: id_col})

        X = df.drop(columns=[id_col, label_col], errors="ignore")

        numeric_cols, cat_cols = [], []
        for c in X.columns:
            s = pd.to_numeric(X[c], errors="coerce")
            if np.isfinite(s.values).mean() >= 0.8:
                numeric_cols.append(c)
            else:
                cat_cols.append(c)

        self.numeric_cols = numeric_cols
        self.cat_cols = cat_cols

        X_num = X[numeric_cols].copy() if numeric_cols else pd.DataFrame(index=X.index)
        for c in numeric_cols:
            X_num[c] = pd.to_numeric(X_num[c], errors="coerce").fillna(0.0).astype(np.float32)

        self.scaler = StandardScaler() if numeric_cols else None
        if self.scaler is not None:
            self.scaler.fit(X_num.values)

        X_cat = X[cat_cols].copy() if cat_cols else pd.DataFrame(index=X.index)
        for c in cat_cols:
            X_cat[c] = X_cat[c].astype(str).fillna("NA")

        self.ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False) if cat_cols else None
        if self.ohe is not None:
            self.ohe.fit(X_cat.values)

        fn = []
        if numeric_cols:
            fn += [f"num::{c}" for c in numeric_cols]
        if self.ohe is not None:
            fn += [f"cat::{x}" for x in self.ohe.get_feature_names_out(cat_cols).tolist()]
        self.feature_names_ = fn

    def transform(self, df: pd.DataFrame, id_col: str, label_col: str):
        df = df.copy()
        df.columns = [c.strip().replace('"','') for c in df.columns]

        if id_col not in df.columns:
            cand = [c for c in df.columns if c.lower().strip() == id_col.lower().strip()]
            if len(cand) == 1:
                df = df.rename(columns={cand[0]: id_col})

        y = df[label_col].values.astype(int)
        X = df.drop(columns=[id_col, label_col], errors="ignore")

        if self.numeric_cols:
            X_num = X[self.numeric_cols].copy()
            for c in self.numeric_cols:
                X_num[c] = pd.to_numeric(X_num[c], errors="coerce").fillna(0.0).astype(np.float32)
            X_num = self.scaler.transform(X_num.values) if self.scaler is not None else X_num.values
        else:
            X_num = np.zeros((len(df), 0), dtype=np.float32)

        if self.ohe is not None and self.cat_cols:
            X_cat = X[self.cat_cols].copy()
            for c in self.cat_cols:
                X_cat[c] = X_cat[c].astype(str).fillna("NA")
            X_cat = self.ohe.transform(X_cat.values).astype(np.float32)
        else:
            X_cat = np.zeros((len(df), 0), dtype=np.float32)

        X_all = np.concatenate([X_num.astype(np.float32), X_cat.astype(np.float32)], axis=1)
        return X_all, y, self.feature_names_

# ============================================================
# Dataset
# ============================================================
class MultiD4CAD_MILDualDataset(Dataset):
    def __init__(self, df_subset: pd.DataFrame, cache_dual: dict, clinical_processor: ClinicalProcessor):
        self.df = df_subset.copy().reset_index(drop=True)
        self.cache = cache_dual
        self.cp = clinical_processor

        keep = []
        for i in range(len(self.df)):
            pid = str(self.df.loc[i, CONFIG["patient_id_col"]])
            if pid in self.cache:
                keep.append(i)
        self.df = self.df.loc[keep].reset_index(drop=True)

        Xc, y, _ = self.cp.transform(self.df, CONFIG["patient_id_col"], CONFIG["label_col"])
        self.Xc = Xc.astype(np.float32)
        self.y = y.astype(int)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        pid = str(self.df.loc[idx, CONFIG["patient_id_col"]])
        y = int(self.y[idx])
        clin = self.Xc[idx]
        eat = self.cache[pid]["eat"]
        pat = self.cache[pid]["pat"]

        return (
            torch.tensor(eat, dtype=torch.float32),
            torch.tensor(pat, dtype=torch.float32),
            torch.tensor(clin, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
        )

# ============================================================
# Models (NO INTERNET)
# ============================================================
def _load_torchvision_resnet_weights(model: nn.Module, pth_path: str):
    if pth_path is None or (not os.path.exists(pth_path)):
        print(f"⚠️ ResNet weights introuvables -> training from scratch. Path={pth_path}")
        return model

    ckpt = torch.load(pth_path, map_location="cpu")

    if isinstance(ckpt, dict) and "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
        sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt and isinstance(ckpt["model"], dict):
        sd = ckpt["model"]
    elif isinstance(ckpt, dict):
        sd = ckpt
    else:
        raise ValueError("Format checkpoint non supporté pour ResNet .pth")

    new_sd = {}
    for k, v in sd.items():
        kk = k
        if kk.startswith("module."):
            kk = kk[len("module."):]
        if kk.startswith("backbone."):
            kk = kk[len("backbone."):]
        if kk.startswith("encoder."):
            kk = kk[len("encoder."):]
        if kk.startswith("fc."):
            continue
        new_sd[kk] = v

    missing, unexpected = model.load_state_dict(new_sd, strict=False)
    print(f"✅ ResNet local chargé: {pth_path}")
    if len(missing) > 0:
        print("   - missing keys (ok si fc.*):", missing[:10], ("..." if len(missing) > 10 else ""))
    if len(unexpected) > 0:
        print("   - unexpected keys:", unexpected[:10], ("..." if len(unexpected) > 10 else ""))
    return model

def build_resnet(backbone="resnet18"):
    if backbone == "resnet18":
        net = models.resnet18(weights=None)
        feat_dim = 512
        net = _load_torchvision_resnet_weights(net, CONFIG.get("resnet18_local_pth", None))
    else:
        net = models.resnet50(weights=None)
        feat_dim = 2048
    net.fc = nn.Identity()
    return net, feat_dim

class ResNetEncoder(nn.Module):
    def __init__(self, out_dim=128, backbone="resnet18", dropout=0.5):
        super().__init__()
        net, feat_dim = build_resnet(backbone)
        self.backbone = net
        self.proj = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, out_dim),
            nn.GELU(),
            nn.LayerNorm(out_dim),
        )

    def forward(self, x):
        f = self.backbone(x)
        return self.proj(f)

class GatedAttentionMIL(nn.Module):
    def __init__(self, dim, attn_dim=128):
        super().__init__()
        self.V = nn.Linear(dim, attn_dim)
        self.U = nn.Linear(dim, attn_dim)
        self.w = nn.Linear(attn_dim, 1)

    def forward(self, H):
        A = torch.tanh(self.V(H)) * torch.sigmoid(self.U(H))
        A = self.w(A).squeeze(-1)
        alpha = torch.softmax(A, dim=1)
        M = torch.sum(alpha.unsqueeze(-1) * H, dim=1)
        return M, alpha

class DualMIL_ClinModel(nn.Module):
    def __init__(self, clin_dim, img_dim=128, backbone="resnet18", dropout=0.5):
        super().__init__()
        self.enc = ResNetEncoder(out_dim=img_dim, backbone=backbone, dropout=dropout)
        self.mil_eat = GatedAttentionMIL(img_dim, attn_dim=128)
        self.mil_pat = GatedAttentionMIL(img_dim, attn_dim=128)

        self.clin_net = nn.Sequential(
            nn.Linear(clin_dim, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),
        )

        fusion_dim = img_dim * 2 + 128
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(256),
            nn.Linear(256, 1),
        )

    def forward(self, eat_bag, pat_bag, clin):
        B, N, C, H, W = eat_bag.shape
        eat_flat = eat_bag.view(B*N, C, H, W)
        pat_flat = pat_bag.view(B*N, C, H, W)

        He = self.enc(eat_flat).view(B, N, -1)
        Hp = self.enc(pat_flat).view(B, N, -1)

        Me, _ = self.mil_eat(He)
        Mp, _ = self.mil_pat(Hp)

        Cc = self.clin_net(clin)
        fused = torch.cat([Me, Mp, Cc], dim=1)
        return self.head(fused)

class BCESmooth(nn.Module):
    def __init__(self, pos_weight=None, smoothing=0.03):
        super().__init__()
        self.pos_weight = pos_weight
        self.smoothing = smoothing

    def forward(self, logits, y):
        y = y * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return nn.functional.binary_cross_entropy_with_logits(
            logits, y, pos_weight=self.pos_weight
        )

# ============================================================
# Cache builder
# ============================================================
def build_cache_dual_2p5d(df_all: pd.DataFrame):
    cache_file = CONFIG["cache_file"]
    if os.path.exists(cache_file):
        print(f"📦 Cache trouvé: {cache_file}")
        with open(cache_file, "rb") as f:
            cache = pickle.load(f)
        print(f"✅ Cache chargé: {len(cache)} patients")
        return cache

    print("🔄 Construction du cache 2.5D (EAT + PAT)...")

    root = Path(CONFIG["data_root"])
    eat_root = root / CONFIG["eat_base_dir"]
    pat_root = root / CONFIG["pat_base_dir"]

    cache = {}
    for pid in tqdm(df_all[CONFIG["patient_id_col"]].astype(str).tolist(), desc="Building cache"):
        try:
            eat_dir = eat_root / pid
            pat_dir = pat_root / pid

            eat_img_p = eat_dir / CONFIG["eat_img"]
            eat_msk_p = eat_dir / CONFIG["eat_msk"]
            pat_img_p = pat_dir / CONFIG["pat_img"]
            pat_msk_p = pat_dir / CONFIG["pat_msk"]

            if not all(p.exists() for p in [eat_img_p, eat_msk_p, pat_img_p, pat_msk_p]):
                continue

            eat_vol, _ = load_nifti(eat_img_p)
            eat_msk, _ = load_nifti(eat_msk_p)
            pat_vol, _ = load_nifti(pat_img_p)
            pat_msk, _ = load_nifti(pat_msk_p)

            eat_inst = build_2p5d_instances(eat_vol, eat_msk, CONFIG["num_instances"], CONFIG["target_hw"], patient_id=pid)
            pat_inst = build_2p5d_instances(pat_vol, pat_msk, CONFIG["num_instances"], CONFIG["target_hw"], patient_id=pid)

            cache[pid] = {"eat": eat_inst.astype(np.float32), "pat": pat_inst.astype(np.float32)}
        except Exception:
            continue

    with open(cache_file, "wb") as f:
        pickle.dump(cache, f)

    print(f"✅ Cache construit: {len(cache)} patients | saved: {cache_file}")
    return cache

# ============================================================
# Load data
# ============================================================
set_seed(42)
csv_path = Path(CONFIG["data_root"]) / CONFIG["clinical_csv"]
df_all = pd.read_csv(csv_path, sep=";")
df_all.columns = [c.strip().replace('"','') for c in df_all.columns]

if CONFIG["patient_id_col"] not in df_all.columns:
    cand = [c for c in df_all.columns if c.lower().strip() == CONFIG["patient_id_col"].lower().strip()]
    if len(cand) == 1:
        df_all = df_all.rename(columns={cand[0]: CONFIG["patient_id_col"]})

if CONFIG["label_col"] not in df_all.columns:
    for c in df_all.columns:
        if "CAD" in c.upper() and ("LABEL" in c.upper() or "CAD" in c.upper()):
            df_all = df_all.rename(columns={c: CONFIG["label_col"]})
            break

df_all[CONFIG["patient_id_col"]] = df_all[CONFIG["patient_id_col"]].astype(str)
df_all[CONFIG["label_col"]] = df_all[CONFIG["label_col"]].apply(binarize_label).astype(int)

print(f"📊 Données cliniques: {len(df_all)} patients | CAD+ {(df_all[CONFIG['label_col']]==1).sum()} | CAD- {(df_all[CONFIG['label_col']]==0).sum()}")

cache_dual = build_cache_dual_2p5d(df_all)

df_all = df_all[df_all[CONFIG["patient_id_col"]].isin(set(cache_dual.keys()))].reset_index(drop=True)
print(f"✅ Dataset final: {len(df_all)} patients avec cache | CAD+ {(df_all[CONFIG['label_col']]==1).sum()} | CAD- {(df_all[CONFIG['label_col']]==0).sum()}")

# ============================================================
# Train / Predict / Calibration
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, use_amp=True):
    model.train()
    total, n = 0.0, 0
    for eat, pat, clin, y in loader:
        eat = eat.to(device)
        pat = pat.to(device)
        clin = clin.to(device)
        y = y.to(device).view(-1, 1)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None and use_amp and device.type == "cuda":
            with torch.amp.autocast(device_type="cuda", enabled=True):
                logits = model(eat, pat, clin)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(eat, pat, clin)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            optimizer.step()

        total += float(loss.item()) * eat.size(0)
        n += eat.size(0)
    return total / max(n, 1)

@torch.no_grad()
def predict_logits(model, loader, device, use_amp=True):
    model.eval()
    logits_all, y_all = [], []
    for eat, pat, clin, y in loader:
        eat = eat.to(device)
        pat = pat.to(device)
        clin = clin.to(device)

        if device.type == "cuda" and use_amp:
            with torch.amp.autocast(device_type="cuda", enabled=True):
                logits = model(eat, pat, clin)
        else:
            logits = model(eat, pat, clin)

        logits_all.append(logits.detach().cpu().numpy().reshape(-1))
        y_all.append(y.numpy().astype(int).reshape(-1))

    logits_all = np.concatenate(logits_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    logits_all = np.nan_to_num(logits_all, nan=0.0, posinf=0.0, neginf=0.0)
    return logits_all, y_all

def platt_calibrate(val_logits, val_y):
    lr = LogisticRegression(solver="lbfgs", max_iter=200)
    lr.fit(val_logits.reshape(-1, 1), val_y.astype(int))
    return lr

def apply_calibrator(calib, logits):
    p = calib.predict_proba(logits.reshape(-1,1))[:,1]
    return np.nan_to_num(p, nan=0.5, posinf=1.0, neginf=0.0)

# ============================================================
# Bootstrap helpers (publication)
# ============================================================
def bootstrap_ci_auc(y_true, y_prob, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        yp = y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        aucs.append(roc_auc_score(yt, yp))
    aucs = np.sort(np.array(aucs, dtype=np.float32))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

def bootstrap_delta_auc(y_true, p_full, p_base, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    p_full = np.asarray(p_full).astype(float)
    p_base = np.asarray(p_base).astype(float)
    n = len(y_true)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue
        deltas.append(roc_auc_score(yt, p_full[idx]) - roc_auc_score(yt, p_base[idx]))
    deltas = np.sort(np.array(deltas, dtype=np.float32))
    return float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5)), float(np.mean(deltas))

# ============================================================
# Clinical-only baseline (OOF)
# ============================================================
def run_clinical_only_oof():
    print("\n" + "="*70)
    print("🧪 BASELINE: CLINICAL-ONLY (OOF)")
    print("="*70)

    y_all = df_all[CONFIG["label_col"]].values.astype(int)
    skf = StratifiedKFold(n_splits=CONFIG["num_folds"], shuffle=True, random_state=42)

    oof_prob = np.full(len(df_all), np.nan, dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(df_all)), y_all), start=1):
        df_tr = df_all.iloc[tr_idx].reset_index(drop=True)
        df_va = df_all.iloc[va_idx].reset_index(drop=True)

        cp = ClinicalProcessor()
        cp.fit(df_tr, CONFIG["patient_id_col"], CONFIG["label_col"])

        Xtr, ytr, _ = cp.transform(df_tr, CONFIG["patient_id_col"], CONFIG["label_col"])
        Xva, yva, _ = cp.transform(df_va, CONFIG["patient_id_col"], CONFIG["label_col"])

        clf = LogisticRegression(max_iter=1000, solver="lbfgs")
        clf.fit(Xtr, ytr)
        pva = clf.predict_proba(Xva)[:, 1].astype(np.float32)

        oof_prob[va_idx] = pva

        auc_fold = roc_auc_score(yva, pva) if len(np.unique(yva)) > 1 else 0.5
        print(f"Fold {fold}: AUC={auc_fold:.4f}")

    valid = ~np.isnan(oof_prob)
    y = y_all[valid]
    p = oof_prob[valid].astype(float)

    auc_oof = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
    ap_oof  = average_precision_score(y, p)
    ci_lo, ci_hi = bootstrap_ci_auc(y, p, n_boot=5000, seed=42)

    print("\n" + "="*70)
    print("📌 CLINICAL-ONLY OOF (publication-ready)")
    print("="*70)
    print(f"AUC-ROC OOF = {auc_oof:.4f} | 95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"AUC-PR  OOF = {ap_oof:.4f}")
    print("="*70)

    out_csv = os.path.join(CONFIG["output_dir"], "oof_clinical_only.csv")
    pd.DataFrame({
        "PatientID": df_all[CONFIG["patient_id_col"]].values.astype(str),
        "y_true": y_all,
        "prob_clinical": oof_prob
    }).to_csv(out_csv, index=False)
    print("✅ Clinical-only OOF saved:", out_csv)

    return oof_prob

# ============================================================
# FULL MODEL (publication-friendly training)
# ============================================================
def run_full_model():
    print("\n" + "="*70)
    print("🚀 FULL MODEL (SAFE COMMIT, local pretrained OK)")
    print("="*70)

    y_all = df_all[CONFIG["label_col"]].values.astype(int)
    skf = StratifiedKFold(n_splits=CONFIG["num_folds"], shuffle=True, random_state=42)

    oof_prob = np.full(len(df_all), np.nan, dtype=np.float32)
    oof_y    = y_all.copy()
    oof_fold = np.zeros(len(df_all), dtype=int)
    patient_ids = df_all[CONFIG["patient_id_col"]].values.astype(str)

    fold_rows = []

    # stabilizers
    patience = 8
    min_delta = 1e-4
    sched_patience = 3
    sched_factor = 0.5

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(df_all)), y_all), start=1):
        print(f"\n{'='*70}\nFOLD {fold}/{CONFIG['num_folds']}\n{'='*70}")

        df_tr = df_all.iloc[tr_idx].reset_index(drop=True)
        df_va = df_all.iloc[va_idx].reset_index(drop=True)

        cp = ClinicalProcessor()
        cp.fit(df_tr, CONFIG["patient_id_col"], CONFIG["label_col"])
        clin_dim = len(cp.feature_names_)

        ds_tr = MultiD4CAD_MILDualDataset(df_tr, cache_dual, cp)
        ds_va = MultiD4CAD_MILDualDataset(df_va, cache_dual, cp)

        ytr = ds_tr.y.astype(int)
        class_counts = np.bincount(ytr, minlength=2).astype(np.float32)
        w0 = 1.0 / max(class_counts[0], 1.0)
        w1 = 1.0 / max(class_counts[1], 1.0)
        weights = np.array([w0 if yy==0 else w1 for yy in ytr], dtype=np.float32)

        gen = torch.Generator()
        gen.manual_seed(12345 + fold)
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True, generator=gen)

        tr_loader = DataLoader(ds_tr, batch_size=CONFIG["batch_size"], sampler=sampler, num_workers=2, pin_memory=True)
        va_loader = DataLoader(ds_va, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

        probs_ens = []

        for seed in CONFIG["seeds"]:
            set_seed(seed + fold)

            model = DualMIL_ClinModel(
                clin_dim=clin_dim,
                img_dim=128,
                backbone="resnet18",
                dropout=CONFIG["dropout"]
            ).to(CONFIG["device"])

            # sampler ON => pos_weight OFF
            criterion = BCESmooth(pos_weight=None, smoothing=0.03)

            optimizer = optim.AdamW(
                model.parameters(),
                lr=CONFIG["lr"],
                weight_decay=CONFIG["weight_decay"]
            )

            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="max", factor=sched_factor, patience=sched_patience
            )


            scaler = torch.amp.GradScaler(enabled=(CONFIG["device"].type=="cuda" and CONFIG["use_amp"]))

            best_auc = -1e9
            best_state = None
            best_ep = 0
            bad_epochs = 0

            for ep in range(1, CONFIG["epochs"] + 1):
                tr_loss = train_one_epoch(
                    model, tr_loader, optimizer, criterion, CONFIG["device"],
                    scaler=scaler, use_amp=CONFIG["use_amp"]
                )

                val_logits, val_y = predict_logits(model, va_loader, CONFIG["device"], use_amp=CONFIG["use_amp"])
                calib = platt_calibrate(val_logits, val_y)
                val_prob = apply_calibrator(calib, val_logits)

                auc_val = roc_auc_score(val_y, val_prob) if len(np.unique(val_y)) > 1 else 0.5
                scheduler.step(auc_val)

                if auc_val > best_auc + min_delta and np.isfinite(tr_loss):
                    best_auc = float(auc_val)
                    best_ep = ep
                    best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                    bad_epochs = 0
                else:
                    bad_epochs += 1

                if ep % 5 == 0 or ep == 1:
                    lr_now = optimizer.param_groups[0]["lr"]
                    print(f"[seed {seed}] E{ep:02d} | Loss {tr_loss:.4f} | AUC {auc_val:.4f} | lr {lr_now:.2e}")

                if bad_epochs >= patience:
                    print(f"[seed {seed}] ⏹ Early stop @E{ep:02d} | best AUC {best_auc:.4f} @E{best_ep}")
                    break

            if best_state is None:
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

            model.load_state_dict({k: v.to(CONFIG["device"]) for k, v in best_state.items()})

            # ============================================================
            # 💾 SAUVEGARDER LE MODÈLE
            # ============================================================
            model_save_dir = os.path.join(CONFIG["output_dir"], "trained_models")
            os.makedirs(model_save_dir, exist_ok=True)
            
            model_filename = f"full_model_fold{fold}_seed{seed}.pth"
            model_save_path = os.path.join(model_save_dir, model_filename)
            
            torch.save({
                'model_state_dict': best_state,
                'fold': fold,
                'seed': seed,
                'best_auc': best_auc,
                'best_epoch': best_ep,
                'clin_dim': clin_dim,
                'config': {
                    'img_dim': 128,
                    'backbone': 'resnet18',
                    'dropout': CONFIG["dropout"]
                }
            }, model_save_path)
            
            print(f"    💾 Model saved: {model_filename} (best AUC={best_auc:.4f})")
            # ============================================================

            val_logits, val_y = predict_logits(model, va_loader, CONFIG["device"], use_amp=CONFIG["use_amp"])
            calib = platt_calibrate(val_logits, val_y)
            val_prob = apply_calibrator(calib, val_logits)

            probs_ens.append(val_prob.astype(np.float32))

        prob_mean = np.mean(np.stack(probs_ens, axis=0), axis=0)
        oof_prob[va_idx] = prob_mean.astype(np.float32)
        oof_fold[va_idx] = fold

        y_true = ds_va.y.astype(int)
        best_thr = search_best_threshold(y_true, prob_mean)  # reporting only
        thr_star = best_thr["thr"]
        m = best_thr["metrics"]

        print(f"\n✅ Fold {fold} Ensemble: AUC={m['auc']:.4f} | F1={m['f1']:.4f} | Thr*={thr_star:.3f}")

        fold_rows.append({
            "fold": fold,
            "auc": float(m["auc"]),
            "f1": float(m["f1"]),
            "precision": float(m["prec"]),
            "recall": float(m["rec"]),
            "specificity": float(m["spec"]),
            "threshold": float(thr_star)
        })

    df_res = pd.DataFrame(fold_rows)
    print("\n" + "="*70)
    print("📊 RÉSUMÉ FULL MODEL")
    print("="*70)
    print(df_res)
    print(f"\n✅ AUC MOYEN: {df_res['auc'].mean():.4f} ± {df_res['auc'].std():.4f}")
    print(f"✅ F1  MOYEN: {df_res['f1'].mean():.4f} ± {df_res['f1'].std():.4f}")

    df_res.to_csv(os.path.join(CONFIG["output_dir"], "full_model_results.csv"), index=False)

    df_oof = pd.DataFrame({
        "PatientID": patient_ids,
        "fold": oof_fold,
        "y_true": oof_y,
        "prob_full": oof_prob
    })
    out_oof = os.path.join(CONFIG["output_dir"], "oof_full_model.csv")
    df_oof.to_csv(out_oof, index=False)
    print("✅ OOF saved:", out_oof)

    # OOF GLOBAL
    df_oof_valid = df_oof.dropna()
    y_true = df_oof_valid["y_true"].values.astype(int)
    y_prob = df_oof_valid["prob_full"].values.astype(float)

    auc_oof = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5
    ap_oof  = average_precision_score(y_true, y_prob)

    ci_lo, ci_hi = bootstrap_ci_auc(y_true, y_prob, n_boot=5000, seed=42)

    best = search_best_threshold(y_true, y_prob)
    thr_oof = best["thr"]
    m_oof = best["metrics"]

    thr_mean = float(df_res["threshold"].mean())
    thr_std  = float(df_res["threshold"].std())

    print("\n" + "="*70)
    print("📌 OOF GLOBAL (PUBLICATION-READY)")
    print("="*70)
    print(f"N samples   = {len(y_true)}")
    print(f"N positives = {int(np.sum(y_true))} ({100*np.mean(y_true):.1f}%)")
    print(f"AUC-ROC OOF = {auc_oof:.4f} | 95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"AUC-PR  OOF = {ap_oof:.4f}")
    print(f"Threshold global OOF = {thr_oof:.3f}")
    print(f"Threshold mean folds = {thr_mean:.3f} ± {thr_std:.3f}")
    print(f"Metrics @ thr={thr_oof:.3f}: F1={m_oof['f1']:.4f} | Prec={m_oof['prec']:.4f} | Rec={m_oof['rec']:.4f} | Spec={m_oof['spec']:.4f}")
    print("="*70)

    df_global = pd.DataFrame([
        {"metric": "AUC-ROC", "value": float(auc_oof), "ci_lower": float(ci_lo), "ci_upper": float(ci_hi), "n": int(len(y_true))},
        {"metric": "AUC-PR",  "value": float(ap_oof),  "ci_lower": np.nan,       "ci_upper": np.nan,       "n": int(len(y_true))},
    ])
    global_csv = os.path.join(CONFIG["output_dir"], "oof_global_metrics.csv")
    df_global.to_csv(global_csv, index=False)
    print("✅ Global metrics saved:", global_csv)

    return df_res, df_oof

# ============================================================
# RUN: Full + Clinical + ΔAUC
# ============================================================
set_seed(42)

results_full, oof_full_df = run_full_model()
oof_clin = run_clinical_only_oof()

valid = (~oof_full_df["prob_full"].isna().values) & (~np.isnan(oof_clin))
y = oof_full_df["y_true"].values.astype(int)[valid]
p_full = oof_full_df["prob_full"].values.astype(float)[valid]
p_clin = oof_clin.astype(float)[valid]

auc_full = roc_auc_score(y, p_full) if len(np.unique(y)) > 1 else 0.5
auc_clin = roc_auc_score(y, p_clin) if len(np.unique(y)) > 1 else 0.5
dauc = auc_full - auc_clin

dlo, dhi, dmean = bootstrap_delta_auc(y, p_full, p_clin, n_boot=5000, seed=42)

print("\n" + "="*70)
print("📌 ΔAUC (FULL - CLINICAL) (BOOTSTRAP)")
print("="*70)
print(f"AUC full     = {auc_full:.4f}")
print(f"AUC clinical = {auc_clin:.4f}")
print(f"ΔAUC         = {dauc:.4f} | 95% CI [{dlo:.4f}, {dhi:.4f}] | mean_boot {dmean:.4f}")
print("="*70)

delta_csv = os.path.join(CONFIG["output_dir"], "delta_auc_full_vs_clinical.csv")
pd.DataFrame([{
    "auc_full": float(auc_full),
    "auc_clinical": float(auc_clin),
    "delta_auc": float(dauc),
    "ci_lower": float(dlo),
    "ci_upper": float(dhi),
    "n": int(len(y))
}]).to_csv(delta_csv, index=False)
print("✅ ΔAUC saved:", delta_csv)

merged_csv = os.path.join(CONFIG["output_dir"], "oof_full_plus_clinical.csv")
tmp = oof_full_df.copy()
tmp["prob_clinical"] = oof_clin
tmp.to_csv(merged_csv, index=False)
print("✅ Merged OOF saved:", merged_csv)

################################
# ============================================================
# 📌 COMPLETE ABLATION STUDY (7 MODELS)
# À ajouter après: print("✅ Merged OOF saved:", merged_csv)
# ============================================================

print("\n" + "="*70)
print("🧪 RUNNING COMPLETE ABLATION STUDY (7 models)")
print("  This will take ~3 hours. Please be patient...")
print("="*70)

# ============================================================
# ARCHITECTURES SPÉCIALISÉES
# ============================================================

# Architecture 1: Single Stream MIL (EAT-only ou PAT-only)
class SingleStreamMIL(nn.Module):
    """MIL avec un seul flux (EAT ou PAT)"""
    def __init__(self, stream="eat", img_dim=128, backbone="resnet18", dropout=0.5):
        super().__init__()
        self.stream = stream
        self.enc = ResNetEncoder(out_dim=img_dim, backbone=backbone, dropout=dropout)
        self.mil = GatedAttentionMIL(img_dim, attn_dim=128)
        
        self.head = nn.Sequential(
            nn.Linear(img_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
    
    def forward(self, eat_bag, pat_bag, clin):
        bag = eat_bag if self.stream == "eat" else pat_bag
        B, N, C, H, W = bag.shape
        bag_flat = bag.view(B*N, C, H, W)
        
        H_feats = self.enc(bag_flat).view(B, N, -1)
        M, _ = self.mil(H_feats)
        
        return self.head(M)

# Architecture 2: Single Stream + Clinical
class SingleStream_ClinModel(nn.Module):
    """MIL single stream avec clinical features"""
    def __init__(self, stream="eat", clin_dim=10, img_dim=128, backbone="resnet18", dropout=0.5):
        super().__init__()
        self.stream = stream
        self.enc = ResNetEncoder(out_dim=img_dim, backbone=backbone, dropout=dropout)
        self.mil = GatedAttentionMIL(img_dim, attn_dim=128)
        
        self.clin_net = nn.Sequential(
            nn.Linear(clin_dim, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),
        )
        
        fusion_dim = img_dim + 128
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(256),
            nn.Linear(256, 1),
        )
    
    def forward(self, eat_bag, pat_bag, clin):
        bag = eat_bag if self.stream == "eat" else pat_bag
        B, N, C, H, W = bag.shape
        bag_flat = bag.view(B*N, C, H, W)
        
        H_feats = self.enc(bag_flat).view(B, N, -1)
        M, _ = self.mil(H_feats)
        
        Cc = self.clin_net(clin)
        fused = torch.cat([M, Cc], dim=1)
        
        return self.head(fused)

# Architecture 3: Dual Stream No Clinical
class DualMIL_NoClinical(nn.Module):
    """Dual MIL (EAT+PAT) sans clinical"""
    def __init__(self, img_dim=128, backbone="resnet18", dropout=0.5):
        super().__init__()
        self.enc = ResNetEncoder(out_dim=img_dim, backbone=backbone, dropout=dropout)
        self.mil_eat = GatedAttentionMIL(img_dim, attn_dim=128)
        self.mil_pat = GatedAttentionMIL(img_dim, attn_dim=128)
        
        fusion_dim = img_dim * 2
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(256),
            nn.Linear(256, 1),
        )
    
    def forward(self, eat_bag, pat_bag, clin):
        B, N, C, H, W = eat_bag.shape
        eat_flat = eat_bag.view(B*N, C, H, W)
        pat_flat = pat_bag.view(B*N, C, H, W)
        
        He = self.enc(eat_flat).view(B, N, -1)
        Hp = self.enc(pat_flat).view(B, N, -1)
        
        Me, _ = self.mil_eat(He)
        Mp, _ = self.mil_pat(Hp)
        
        fused = torch.cat([Me, Mp], dim=1)
        return self.head(fused)

# ============================================================
# FONCTION GÉNÉRIQUE D'ENTRAÎNEMENT
# ============================================================
def run_ablation_model(model_factory, name, show_progress=True):
    """
    Entraîne un modèle d'ablation avec CV 5-fold + ensemble 3 seeds
    
    Args:
        model_factory: function(clin_dim) -> nn.Module
        name: str (nom du modèle pour affichage)
        show_progress: bool (afficher détails training)
    
    Returns:
        oof_prob: np.array (OOF predictions)
    """
    if show_progress:
        print(f"\n{'='*70}")
        print(f"🔬 ABLATION: {name}")
        print(f"{'='*70}")
    else:
        print(f"\n🔬 {name}...", end=" ", flush=True)
    
    y_all = df_all[CONFIG["label_col"]].values.astype(int)
    skf = StratifiedKFold(n_splits=CONFIG["num_folds"], shuffle=True, random_state=42)
    
    oof_prob = np.full(len(df_all), np.nan, dtype=np.float32)
    
    # Training params (identiques au Full Model)
    patience = 8
    min_delta = 1e-4
    sched_patience = 3
    sched_factor = 0.5
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(df_all)), y_all), start=1):
        if not show_progress:
            print(f"F{fold}", end="", flush=True)
        
        df_tr = df_all.iloc[tr_idx].reset_index(drop=True)
        df_va = df_all.iloc[va_idx].reset_index(drop=True)
        
        # Clinical processor
        cp = ClinicalProcessor()
        cp.fit(df_tr, CONFIG["patient_id_col"], CONFIG["label_col"])
        clin_dim = len(cp.feature_names_)
        
        # Datasets
        ds_tr = MultiD4CAD_MILDualDataset(df_tr, cache_dual, cp)
        ds_va = MultiD4CAD_MILDualDataset(df_va, cache_dual, cp)
        
        # Weighted sampler
        ytr = ds_tr.y.astype(int)
        class_counts = np.bincount(ytr, minlength=2).astype(np.float32)
        w0 = 1.0 / max(class_counts[0], 1.0)
        w1 = 1.0 / max(class_counts[1], 1.0)
        weights = np.array([w0 if yy==0 else w1 for yy in ytr], dtype=np.float32)
        
        gen = torch.Generator()
        gen.manual_seed(12345 + fold)
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True, generator=gen)
        
        # Dataloaders
        tr_loader = DataLoader(ds_tr, batch_size=CONFIG["batch_size"], sampler=sampler, num_workers=2, pin_memory=True)
        va_loader = DataLoader(ds_va, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
        
        # Ensemble 3 seeds
        probs_ens = []
        
        for seed_idx, seed in enumerate(CONFIG["seeds"]):
            set_seed(seed + fold)
            
            # Créer modèle
            model = model_factory(clin_dim).to(CONFIG["device"])
            
            # Loss, optimizer, scheduler
            criterion = BCESmooth(pos_weight=None, smoothing=0.03)
            optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=sched_factor, patience=sched_patience)
            scaler = torch.amp.GradScaler(enabled=(CONFIG["device"].type=="cuda" and CONFIG["use_amp"]))
            
            # Training loop avec early stopping
            best_auc = -1e9
            best_state = None
            bad_epochs = 0
            
            for ep in range(1, CONFIG["epochs"] + 1):
                # Train
                tr_loss = train_one_epoch(model, tr_loader, optimizer, criterion, CONFIG["device"], scaler=scaler, use_amp=CONFIG["use_amp"])
                
                # Validation
                val_logits, val_y = predict_logits(model, va_loader, CONFIG["device"], use_amp=CONFIG["use_amp"])
                calib = platt_calibrate(val_logits, val_y)
                val_prob = apply_calibrator(calib, val_logits)
                
                auc_val = roc_auc_score(val_y, val_prob) if len(np.unique(val_y)) > 1 else 0.5
                scheduler.step(auc_val)
                
                # Early stopping check
                if auc_val > best_auc + min_delta and np.isfinite(tr_loss):
                    best_auc = float(auc_val)
                    best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                    bad_epochs = 0
                else:
                    bad_epochs += 1
                
                if bad_epochs >= patience:
                    if show_progress and seed_idx == 0:  # Afficher seulement pour premier seed
                        print(f"  Fold {fold} seed {seed}: Early stop @E{ep} | best AUC {best_auc:.4f}")
                    break
            
            # Reload best state
            if best_state is None:
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            
            model.load_state_dict({k: v.to(CONFIG["device"]) for k, v in best_state.items()})
            
            # Final predictions
            val_logits, val_y = predict_logits(model, va_loader, CONFIG["device"], use_amp=CONFIG["use_amp"])
            calib = platt_calibrate(val_logits, val_y)
            val_prob = apply_calibrator(calib, val_logits)
            
            probs_ens.append(val_prob.astype(np.float32))
        
        # Moyenne ensemble
        prob_mean = np.mean(np.stack(probs_ens, axis=0), axis=0)
        oof_prob[va_idx] = prob_mean.astype(np.float32)
        
        # Report fold AUC
        auc_fold = roc_auc_score(ds_va.y, prob_mean) if len(np.unique(ds_va.y)) > 1 else 0.5
        if show_progress:
            print(f"  ✅ Fold {fold}: AUC={auc_fold:.4f}")
        else:
            print(f".", end="", flush=True)
    
    # OOF Global metrics
    valid = ~np.isnan(oof_prob)
    y = y_all[valid]
    p = oof_prob[valid].astype(float)
    
    auc_oof = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
    ci_lo, ci_hi = bootstrap_ci_auc(y, p, n_boot=5000)
    
    if show_progress:
        print(f"\n  📌 {name}: AUC={auc_oof:.4f} [95% CI: {ci_lo:.4f}-{ci_hi:.4f}]")
    else:
        print(f" AUC={auc_oof:.4f} ✅")
    
    return oof_prob

# ============================================================
# EXÉCUTER LES 5 NOUVELLES ABLATIONS
# (Clinical Only et FULL déjà faits)
# ============================================================

import time
start_time = time.time()

# 1) EAT Only
oof_eat = run_ablation_model(
    lambda clin_dim: SingleStreamMIL(stream="eat", img_dim=128, backbone="resnet18", dropout=CONFIG["dropout"]),
    "EAT Only",
    show_progress=False
)

# 2) PAT Only
oof_pat = run_ablation_model(
    lambda clin_dim: SingleStreamMIL(stream="pat", img_dim=128, backbone="resnet18", dropout=CONFIG["dropout"]),
    "PAT Only",
    show_progress=False
)

# 3) EAT + Clinical
oof_eat_clin = run_ablation_model(
    lambda clin_dim: SingleStream_ClinModel(stream="eat", clin_dim=clin_dim, img_dim=128, backbone="resnet18", dropout=CONFIG["dropout"]),
    "EAT + Clinical",
    show_progress=False
)

# 4) PAT + Clinical
oof_pat_clin = run_ablation_model(
    lambda clin_dim: SingleStream_ClinModel(stream="pat", clin_dim=clin_dim, img_dim=128, backbone="resnet18", dropout=CONFIG["dropout"]),
    "PAT + Clinical",
    show_progress=False
)

# 5) Imaging Only (EAT+PAT)
oof_imaging = run_ablation_model(
    lambda clin_dim: DualMIL_NoClinical(img_dim=128, backbone="resnet18", dropout=CONFIG["dropout"]),
    "Imaging Only (EAT+PAT)",
    show_progress=False
)

elapsed = (time.time() - start_time) / 60
print(f"\n⏱️  All ablations completed in {elapsed:.1f} minutes")

# ============================================================
# SAUVEGARDER TOUTES LES PRÉDICTIONS OOF
# ============================================================
print("\n" + "="*70)
print("💾 SAVING ALL OOF PREDICTIONS")
print("="*70)

# Récupérer prédictions Full et Clinical (déjà calculées)
oof_full = oof_full_df["prob_full"].values
# oof_clin déjà disponible de ton code original

# Créer DataFrame avec toutes les prédictions
df_oof_all = pd.DataFrame({
    "PatientID": df_all[CONFIG["patient_id_col"]].values.astype(str),
    "y_true": df_all[CONFIG["label_col"]].values.astype(int),
    "prob_full": oof_full,
    "prob_imaging": oof_imaging,
    "prob_eat_clin": oof_eat_clin,
    "prob_pat_clin": oof_pat_clin,
    "prob_eat": oof_eat,
    "prob_pat": oof_pat,
    "prob_clinical": oof_clin,
})

oof_all_csv = os.path.join(CONFIG["output_dir"], "oof_all_ablations.csv")
df_oof_all.to_csv(oof_all_csv, index=False)
print(f"✅ All OOF predictions saved: {oof_all_csv}")

# ============================================================
# TABLEAU ABLATION COMPLET
# ============================================================
print("\n" + "="*70)
print("📊 COMPLETE ABLATION TABLE")
print("="*70)

# Trouver indices valides pour tous les modèles
valid_all = (
    ~np.isnan(oof_full) & 
    ~np.isnan(oof_clin) & 
    ~np.isnan(oof_imaging) &
    ~np.isnan(oof_eat) &
    ~np.isnan(oof_pat) &
    ~np.isnan(oof_eat_clin) &
    ~np.isnan(oof_pat_clin)
)

y = df_all[CONFIG["label_col"]].values.astype(int)[valid_all]

print(f"Valid samples for comparison: {valid_all.sum()} / {len(df_all)}")

# Calculer métriques pour chaque modèle
ablation_results = []

for name, probs in [
    ("FULL (EAT+PAT+Clinical)", oof_full[valid_all]),
    ("Imaging Only (EAT+PAT)", oof_imaging[valid_all]),
    ("EAT + Clinical", oof_eat_clin[valid_all]),
    ("PAT + Clinical", oof_pat_clin[valid_all]),
    ("EAT Only", oof_eat[valid_all]),
    ("PAT Only", oof_pat[valid_all]),
    ("Clinical Only", oof_clin[valid_all]),
]:
    auc = roc_auc_score(y, probs)
    ci_lo, ci_hi = bootstrap_ci_auc(y, probs, n_boot=5000)
    ap = average_precision_score(y, probs)
    
    # Meilleur threshold par F1
    thr, _ = best_threshold_by_f1(y, probs)
    metrics_at_thr = compute_metrics(y, probs, thr)
    
    ablation_results.append({
        "model": name,
        "auc": auc,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "auprc": ap,
        "f1": metrics_at_thr["f1"],
        "precision": metrics_at_thr["prec"],
        "recall": metrics_at_thr["rec"],
        "specificity": metrics_at_thr["spec"],
        "n": len(y)
    })

df_ablation = pd.DataFrame(ablation_results).sort_values("auc", ascending=False).reset_index(drop=True)

print("\n" + df_ablation.to_string(index=False))

abl_csv = os.path.join(CONFIG["output_dir"], "paper_ablation_complete.csv")
df_ablation.to_csv(abl_csv, index=False)
print(f"\n✅ Complete ablation table saved: {abl_csv}")

# ============================================================
# COMPARAISONS STATISTIQUES
# ============================================================
print("\n" + "="*70)
print("📊 STATISTICAL COMPARISONS")
print("="*70)

comparisons = []

print("\n1️⃣  FULL vs All Others:")
print("-" * 70)

# FULL vs tous les autres
for name, probs in [
    ("Clinical Only", oof_clin[valid_all]),
    ("EAT Only", oof_eat[valid_all]),
    ("PAT Only", oof_pat[valid_all]),
    ("EAT + Clinical", oof_eat_clin[valid_all]),
    ("PAT + Clinical", oof_pat_clin[valid_all]),
    ("Imaging Only", oof_imaging[valid_all]),
]:
    delta, lo, hi = bootstrap_delta_auc(y, oof_full[valid_all], probs, n_boot=5000)
    
    # Test significativité
    significant = "***" if (lo > 0 or hi < 0) else ""
    
    comparisons.append({
        "comparison": f"FULL vs {name}",
        "delta_auc": delta,
        "ci_lo": lo,
        "ci_hi": hi,
        "significant": "Yes" if significant else "No"
    })
    
    print(f"  FULL vs {name:25s}: ΔAUC={delta:+.4f} [{lo:+.4f}, {hi:+.4f}] {significant}")

print("\n2️⃣  Key Comparisons:")
print("-" * 70)

# EAT vs PAT
delta, lo, hi = bootstrap_delta_auc(y, oof_eat[valid_all], oof_pat[valid_all], n_boot=5000)
significant = "***" if (lo > 0 or hi < 0) else ""
comparisons.append({
    "comparison": "EAT Only vs PAT Only",
    "delta_auc": delta,
    "ci_lo": lo,
    "ci_hi": hi,
    "significant": "Yes" if significant else "No"
})
print(f"  EAT vs PAT:                    ΔAUC={delta:+.4f} [{lo:+.4f}, {hi:+.4f}] {significant}")

# EAT+Clin vs Clinical
delta, lo, hi = bootstrap_delta_auc(y, oof_eat_clin[valid_all], oof_clin[valid_all], n_boot=5000)
significant = "***" if lo > 0 else ""
comparisons.append({
    "comparison": "EAT+Clin vs Clinical",
    "delta_auc": delta,
    "ci_lo": lo,
    "ci_hi": hi,
    "significant": "Yes" if significant else "No"
})
print(f"  EAT+Clin vs Clinical:          ΔAUC={delta:+.4f} [{lo:+.4f}, {hi:+.4f}] {significant}")

# PAT+Clin vs Clinical
delta, lo, hi = bootstrap_delta_auc(y, oof_pat_clin[valid_all], oof_clin[valid_all], n_boot=5000)
significant = "***" if lo > 0 else ""
comparisons.append({
    "comparison": "PAT+Clin vs Clinical",
    "delta_auc": delta,
    "ci_lo": lo,
    "ci_hi": hi,
    "significant": "Yes" if significant else "No"
})
print(f"  PAT+Clin vs Clinical:          ΔAUC={delta:+.4f} [{lo:+.4f}, {hi:+.4f}] {significant}")

# Imaging vs Clinical
delta, lo, hi = bootstrap_delta_auc(y, oof_imaging[valid_all], oof_clin[valid_all], n_boot=5000)
significant = "***" if lo > 0 else ""
comparisons.append({
    "comparison": "Imaging vs Clinical",
    "delta_auc": delta,
    "ci_lo": lo,
    "ci_hi": hi,
    "significant": "Yes" if significant else "No"
})
print(f"  Imaging vs Clinical:           ΔAUC={delta:+.4f} [{lo:+.4f}, {hi:+.4f}] {significant}")

# EAT+Clin vs EAT Only
delta, lo, hi = bootstrap_delta_auc(y, oof_eat_clin[valid_all], oof_eat[valid_all], n_boot=5000)
comparisons.append({
    "comparison": "EAT+Clin vs EAT Only",
    "delta_auc": delta,
    "ci_lo": lo,
    "ci_hi": hi,
    "significant": "Yes" if (lo > 0 or hi < 0) else "No"
})
print(f"  EAT+Clin vs EAT Only:          ΔAUC={delta:+.4f} [{lo:+.4f}, {hi:+.4f}]")

# PAT+Clin vs PAT Only
delta, lo, hi = bootstrap_delta_auc(y, oof_pat_clin[valid_all], oof_pat[valid_all], n_boot=5000)
comparisons.append({
    "comparison": "PAT+Clin vs PAT Only",
    "delta_auc": delta,
    "ci_lo": lo,
    "ci_hi": hi,
    "significant": "Yes" if (lo > 0 or hi < 0) else "No"
})
print(f"  PAT+Clin vs PAT Only:          ΔAUC={delta:+.4f} [{lo:+.4f}, {hi:+.4f}]")

print("\n*** = Statistically significant (95% CI excludes 0)")

# Sauvegarder toutes les comparaisons
df_stats = pd.DataFrame(comparisons)
stats_csv = os.path.join(CONFIG["output_dir"], "paper_statistics_complete.csv")
df_stats.to_csv(stats_csv, index=False)
print(f"\n✅ Complete statistics saved: {stats_csv}")

# ============================================================
# FIGURES PUBLICATION-READY
# ============================================================
print("\n" + "="*70)
print("📊 GENERATING PUBLICATION FIGURES")
print("="*70)

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# Figure 1: ROC Curves (tous les modèles)
print("  Generating ROC curves...")
plt.figure(figsize=(10,8), dpi=150)

models_for_plot = [
    ("FULL", oof_full[valid_all], '#e74c3c', 2.5, '-'),
    ("Imaging", oof_imaging[valid_all], '#3498db', 2.5, '-'),
    ("EAT+Clin", oof_eat_clin[valid_all], '#2ecc71', 2.0, '-'),
    ("PAT+Clin", oof_pat_clin[valid_all], '#f39c12', 2.0, '-'),
    ("EAT", oof_eat[valid_all], '#9b59b6', 1.5, '--'),
    ("PAT", oof_pat[valid_all], '#1abc9c', 1.5, '--'),
    ("Clinical", oof_clin[valid_all], '#95a5a6', 1.5, ':'),
]

for name, probs, color, lw, ls in models_for_plot:
    fpr, tpr, _ = roc_curve(y, probs)
    auc = roc_auc_score(y, probs)
    plt.plot(fpr, tpr, linewidth=lw, linestyle=ls, label=f"{name} (AUC={auc:.3f})", color=color)

plt.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
plt.xlabel("False Positive Rate", fontsize=14, fontweight='bold')
plt.ylabel("True Positive Rate", fontsize=14, fontweight='bold')
plt.title("Complete Ablation Study - ROC Curves (n=117)", fontsize=15, fontweight='bold')
plt.legend(loc="lower right", fontsize=11, frameon=True, shadow=True, fancybox=True)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()

roc_png = os.path.join(CONFIG["output_dir"], "paper_roc_complete.png")
plt.savefig(roc_png, dpi=300, bbox_inches="tight")
plt.close()
print(f"  ✅ ROC curves saved: {roc_png}")

# Figure 2: Bar Chart Ablations
print("  Generating ablation bar chart...")
df_sorted = df_ablation.sort_values("auc", ascending=True).copy()

plt.figure(figsize=(12,7), dpi=150)
colors_bar = ['#95a5a6', '#1abc9c', '#9b59b6', '#f39c12', '#2ecc71', '#3498db', '#e74c3c']

y_pos = np.arange(len(df_sorted))
bars = plt.barh(y_pos, df_sorted['auc'], color=colors_bar[:len(df_sorted)], 
                edgecolor='black', linewidth=1.5, alpha=0.85)

# Highlight FULL
full_idx = df_sorted[df_sorted['model'].str.contains('FULL')].index
if len(full_idx) > 0:
    idx = list(df_sorted.index).index(full_idx[0])
    bars[idx].set_edgecolor('#c0392b')
    bars[idx].set_linewidth(3.5)
    bars[idx].set_alpha(1.0)

plt.yticks(y_pos, df_sorted['model'], fontsize=12, fontweight='bold')
plt.xlabel('AUC-ROC', fontsize=14, fontweight='bold')
plt.title('Complete Ablation Study - Model Comparison (n=117)', fontsize=15, fontweight='bold')
plt.axvline(x=0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.6, label='Random')

# Annoter avec AUC + CI
for i, (auc_val, ci_lo, ci_hi) in enumerate(zip(df_sorted['auc'], df_sorted['ci_lo'], df_sorted['ci_hi'])):
    plt.text(auc_val + 0.012, i, f"{auc_val:.3f}\n[{ci_lo:.2f}-{ci_hi:.2f}]", 
            va='center', ha='left', fontsize=10, fontweight='bold')

plt.xlim([0.50, 0.95])
plt.grid(axis='x', alpha=0.3, linestyle='--')
plt.legend(fontsize=11, loc='lower right')
plt.tight_layout()

bar_png = os.path.join(CONFIG["output_dir"], "paper_ablation_bar_complete.png")
plt.savefig(bar_png, dpi=300, bbox_inches="tight")
plt.close()
print(f"  ✅ Ablation bar chart saved: {bar_png}")

# Figure 3: Comparaison EAT vs PAT (subplot)
print("  Generating EAT vs PAT comparison...")
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

# Panel A: ROC
ax = axes[0]
for name, probs, color, lw in [
    ("EAT + Clinical", oof_eat_clin[valid_all], '#2ecc71', 2.5),
    ("PAT + Clinical", oof_pat_clin[valid_all], '#f39c12', 2.5),
    ("EAT Only", oof_eat[valid_all], '#9b59b6', 2.0),
    ("PAT Only", oof_pat[valid_all], '#1abc9c', 2.0),
]:
    fpr, tpr, _ = roc_curve(y, probs)
    auc = roc_auc_score(y, probs)
    ax.plot(fpr, tpr, linewidth=lw, label=f"{name} (AUC={auc:.3f})", color=color)

ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5)
ax.set_xlabel("False Positive Rate", fontsize=12, fontweight='bold')
ax.set_ylabel("True Positive Rate", fontsize=12, fontweight='bold')
ax.set_title("A) EAT vs PAT - ROC Curves", fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3, linestyle='--')

# Panel B: Bar comparison
ax = axes[1]
models_comp = ["EAT+Clin", "PAT+Clin", "EAT", "PAT"]
aucs_comp = [
    roc_auc_score(y, oof_eat_clin[valid_all]),
    roc_auc_score(y, oof_pat_clin[valid_all]),
    roc_auc_score(y, oof_eat[valid_all]),
    roc_auc_score(y, oof_pat[valid_all]),
]
colors_comp = ['#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

bars = ax.barh(models_comp, aucs_comp, color=colors_comp, edgecolor='black', linewidth=1.5, alpha=0.85)
ax.set_xlabel("AUC-ROC", fontsize=12, fontweight='bold')
ax.set_title("B) EAT vs PAT - AUC Comparison", fontsize=13, fontweight='bold')
ax.set_xlim([0.65, 0.90])
ax.axvline(x=0.5, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.grid(axis='x', alpha=0.3, linestyle='--')

for i, (model, auc_val) in enumerate(zip(models_comp, aucs_comp)):
    ax.text(auc_val + 0.005, i, f"{auc_val:.3f}", va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
eatvspat_png = os.path.join(CONFIG["output_dir"], "paper_eat_vs_pat.png")
plt.savefig(eatvspat_png, dpi=300, bbox_inches="tight")
plt.close()
print(f"  ✅ EAT vs PAT comparison saved: {eatvspat_png}")

# ============================================================
# SUMMARY REPORT
# ============================================================
print("\n" + "="*70)
print("🎉 COMPLETE ABLATION STUDY FINISHED")
print("="*70)

print("\n📁 Generated Files:")
print(f"  1. {oof_all_csv}")
print(f"  2. {abl_csv}")
print(f"  3. {stats_csv}")
print(f"  4. {roc_png}")
print(f"  5. {bar_png}")
print(f"  6. {eatvspat_png}")

print("\n📊 Key Findings:")
best_model = df_ablation.iloc[0]
print(f"  • Best model: {best_model['model']}")
print(f"  • Best AUC: {best_model['auc']:.4f} [95% CI: {best_model['ci_lo']:.4f}-{best_model['ci_hi']:.4f}]")

eat_auc = df_ablation[df_ablation['model']=='EAT Only']['auc'].values[0]
pat_auc = df_ablation[df_ablation['model']=='PAT Only']['auc'].values[0]
print(f"  • EAT vs PAT: {eat_auc:.4f} vs {pat_auc:.4f} (EAT {'better' if eat_auc > pat_auc else 'worse'})")

sig_comps = df_stats[df_stats['significant']=='Yes']
print(f"  • Significant comparisons: {len(sig_comps)} / {len(df_stats)}")

print("\n✅ All outputs ready for publication!")
print("="*70)